[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/HW/STAT3009_HW2.ipynb)

# STAT3009 Homework 2 — Neural Networks and Neural Collaborative Filtering

**Coverage:** Neural Networks + Neural Collaborative Filtering  
**Total:** 100 points  
**Due:** Thursday, 3 December 2026, before lecture

This homework has three implementation questions:

```text
Q1  two-branch recommender       → rating regression, RMSE / MAE
Q2  semi-parametric housing model → regression, RMSE / MAE
Q3  custom recommender formula    → rating regression, RMSE / MAE
```

The setup code is provided. Each question asks you to complete a small `nn.Module`, fit it with skorch, and interpret the result. Use **PyTorch** and **skorch** throughout. Use **vanilla SGD**; do not switch to Adam.

## Learning goals

By the end, you should be able to:

1. build a two-branch `nn.Module` with embedding lookups;
2. wrap a custom model in `NeuralNetRegressor` and train it with SGD;
3. translate a mathematical prediction formula into a `forward` pass;
4. evaluate rating predictions with RMSE and MAE;
5. compare a semi-parametric model with linear and plain-MLP baselines.

## Point distribution

| Question | Topic | Points |
|---|---|---:|
| 1 | Two-branch recommender | 35 |
| 2 | Semi-parametric housing model | 30 |
| 3 | Custom recommender | 35 |
|  | **Total** | **100** |

## Data and metric notes

- **Q1 and Q3** use the Netflix 1–5 star split, the same split as HW1 and the NCF notebook. The target is a continuous rating; we report RMSE and MAE.
- **Q2** uses the California housing split. The target is a continuous value, so we report RMSE and MAE.
- The training loss is **MSE** in all three questions. RMSE and MAE are **reporting metrics**, not the training loss.
- Keep the test targets closed until the evaluation cell of each question.

# 0. Setup — run these cells

The following cells load and prepare the two datasets. They are provided and are not graded.

In [ ]:
# In Colab, uncomment the next line if skorch is not installed:
# !pip install skorch

import sys
import warnings

warnings.filterwarnings(
    'ignore', message='The given NumPy array is not writable'
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import skorch
import torch
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from skorch import NeuralNetRegressor
from skorch.dataset import ValidSplit
from torch import nn

pd.set_option('display.max_rows', 12)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
plt.rcParams['figure.dpi'] = 110

torch.manual_seed(0)
np.random.seed(0)

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('skorch:', skorch.__version__)
print('scikit-learn:', sklearn.__version__)
print('NumPy:', np.__version__)

## California housing (Q2)

We standardize the eight numeric features. The test target is opened only in the Q2 evaluation cell.

In [ ]:
HOUSING_BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/housing'
)

house_train = pd.read_csv(f'{HOUSING_BASE_URL}/train.csv')
feature_columns = [
    'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
    'Population', 'AveOccup', 'Latitude', 'Longitude',
]
target_column = 'MedHouseVal'

X_train_house = house_train[feature_columns].to_numpy(dtype=np.float32)
y_train_house = house_train[[target_column]].to_numpy(dtype=np.float32)

house_test_features = pd.read_csv(
    f'{HOUSING_BASE_URL}/test.csv', usecols=feature_columns
)
X_test_house = house_test_features.to_numpy(dtype=np.float32)

house_scaler = StandardScaler()
X_train_house = house_scaler.fit_transform(X_train_house).astype(np.float32)
X_test_house = house_scaler.transform(X_test_house).astype(np.float32)

print('house training:', X_train_house.shape, y_train_house.shape)
print('house test inputs:', X_test_house.shape)
display(house_train.head())

## Rating data (Q1 and Q3)

We use the Netflix 1–5 star split, the same split as HW1. `LabelEncoder` gives each user and item a consecutive integer ID. The test target is opened only in the Q1 evaluation cell.

In [ ]:
RS_BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/netflix'
)

train = pd.read_csv(
    f'{RS_BASE_URL}/train.csv', usecols=['user_id', 'movie_id', 'rating']
)
test_inputs = pd.read_csv(
    f'{RS_BASE_URL}/test.csv', usecols=['user_id', 'movie_id']
)

user_encoder = LabelEncoder().fit(
    pd.concat([train['user_id'], test_inputs['user_id']])
)
item_encoder = LabelEncoder().fit(
    pd.concat([train['movie_id'], test_inputs['movie_id']])
)

X_train = np.column_stack([
    user_encoder.transform(train['user_id']),
    item_encoder.transform(train['movie_id']),
]).astype(np.float32)
X_test = np.column_stack([
    user_encoder.transform(test_inputs['user_id']),
    item_encoder.transform(test_inputs['movie_id']),
]).astype(np.float32)

y_train = train['rating'].to_numpy(dtype=np.float32).reshape(-1, 1)

n_users = len(user_encoder.classes_)
n_items = len(item_encoder.classes_)
mu = float(y_train.mean())

print('recommender training:', X_train.shape, y_train.shape)
print('recommender test inputs:', X_test.shape)
print('users:', n_users, 'items:', n_items)
print('global mean rating:', f'{mu:.4f}')
display(train.head())

The rating data has **51,161 training rows** and **51,161 test input rows**. The rating is a 1–5 star value. A prediction is a rating; we report RMSE and MAE.

# Question 1 — A two-branch recommender [35 points]

We start from the NCF idea: two embedding lookups, then a small MLP. The prediction rule is

$$
\widehat r_{ui}=\mu+a_u+b_i+g_\theta([p_u,q_i]),
$$

where $[p_u,q_i]$ means concatenate the two vectors. The output is a rating; we train with MSE and report RMSE and MAE.

## Q1.1 Complete the two-branch model [10 points]

**Your task:** complete `TwoBranchNCF`.

- In `__init__`, finish `self.net` with one hidden layer and a scalar output.
- In `forward`, concatenate the two embeddings and add the global mean, both biases, and the MLP output.

In [ ]:
class TwoBranchNCF(nn.Module):
    def __init__(self, n_users, n_items, K=16, hidden=32, mu=0.0):
        super().__init__()
        torch.manual_seed(0)
        self.user_emb = nn.Embedding(n_users, K)
        self.item_emb = nn.Embedding(n_items, K)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.mu = nn.Parameter(torch.tensor([mu]))
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

        self.net = nn.Sequential(
            nn.Linear(2 * K, hidden),
            nn.ReLU(),
            # YOUR TASK: add one hidden layer and the scalar output
            ...
        )

    def forward(self, x):
        u = x[:, 0].long()
        i = x[:, 1].long()
        # YOUR TASK: concatenate the two embedding vectors
        z = ...
        # YOUR TASK: return mu + user bias + item bias + MLP(z)
        return ...

### Shape check

After completing the class, run the next cell. The output should be `(5, 1)`.

In [ ]:
demo_two_branch = TwoBranchNCF(n_users, n_items)
print(demo_two_branch)
print('learned numbers:', sum(p.numel() for p in demo_two_branch.parameters()))
print('output shape:', tuple(demo_two_branch(torch.as_tensor(X_train[:5])).shape))

## Q1.2 Fit with skorch [10 points]

**Your task:** complete the skorch wrapper and fit the model.

Use:

- `criterion=nn.MSELoss`
- `optimizer=torch.optim.SGD`
- `optimizer__lr=0.05`
- `max_epochs=30`
- `batch_size=256`
- a fixed `ValidSplit(0.1, random_state=0)`

The three `...` values are the learning rate, number of epochs, and batch size.

In [ ]:
torch.manual_seed(0)
two_branch = NeuralNetRegressor(
    module=TwoBranchNCF,
    module__n_users=n_users,
    module__n_items=n_items,
    module__mu=mu,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=...,          # YOUR TASK
    max_epochs=...,             # YOUR TASK
    batch_size=...,             # YOUR TASK
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)

two_branch.fit(X_train, y_train)
print('epochs run:', len(two_branch.history))
print('final training RMSE:',
      f"{np.sqrt(two_branch.history[-1, 'train_loss']):.4f}")
print('final validation RMSE:',
      f"{np.sqrt(two_branch.history[-1, 'valid_loss']):.4f}")

In [ ]:
epochs = np.arange(1, len(two_branch.history) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, np.sqrt(two_branch.history[:, 'train_loss']),
         label='training RMSE')
plt.plot(epochs, np.sqrt(two_branch.history[:, 'valid_loss']),
         label='validation RMSE')
plt.xlabel('epoch')
plt.ylabel('RMSE')
plt.title('Two-branch recommender: training and validation')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Q1.3 Evaluate with RMSE and MAE [10 points]

**Your task:** open the test target, predict a rating for every test pair, and report **RMSE** and **MAE**.

A reasonable run should give RMSE around **0.96–1.00** and MAE around **0.76–0.80**; exact values depend on the initialization.

In [ ]:
test_labels = pd.read_csv(
    f'{RS_BASE_URL}/test.csv', usecols=['rating']
)
y_test = test_labels['rating'].to_numpy(dtype=np.float32).reshape(-1, 1)

# YOUR TASK: predict a rating for every test pair
two_branch_predictions = ...

# YOUR TASK: compute RMSE and MAE
two_branch_rmse = ...
two_branch_mae = ...

print('two-branch RMSE:', f'{two_branch_rmse:.4f}')
print('two-branch MAE: ', f'{two_branch_mae:.4f}')

## Q1.4 Interpret the two branches [5 points]

Answer in two or three sentences:

1. What does each embedding branch contribute before the concatenation?
2. Why does the model concatenate the two vectors instead of adding them?
3. Why do we use MSE for training but RMSE and MAE for reporting?

**Your answer:**

> Write your answer here.

# Question 2 — Semi-parametric housing model [30 points]

A semi-parametric model adds a linear part and a nonlinear part:

$$
f(\mathbf{x})=\beta^\top\mathbf{x}+h(\mathbf{x}),
$$

where $\beta^\top\mathbf{x}$ is a linear prediction and $h(\cdot)$ is a small neural network. The two branches share the same input and are added at the output.

This is the same additive idea as additive NCF: keep a simple path, then learn a correction.

## Q2.1 Complete the semi-parametric model [10 points]

**Your task:** complete `SemiParametricNet`.

- `self.linear` should be a `nn.Linear(8, 1)` branch.
- `self.nonlinear` should be a small MLP: `8 → 32 → 16 → 1`, with ReLU between hidden layers.
- `forward` should add the two outputs.

In [ ]:
class SemiParametricNet(nn.Module):
    def __init__(self, n_features=8, hidden=32):
        super().__init__()
        torch.manual_seed(0)
        # YOUR TASK: linear branch
        self.linear = ...
        # YOUR TASK: nonlinear branch
        self.nonlinear = ...

    def forward(self, x):
        # YOUR TASK: add the two branches
        return ...

### Shape check

After completing the class, run the next cell. The output should be `(5, 1)`.

In [ ]:
demo_semi = SemiParametricNet()
print(demo_semi)
print('output shape:', tuple(demo_semi(torch.as_tensor(X_train_house[:5])).shape))

## Q2.2 Fit with skorch [8 points]

**Your task:** complete the skorch wrapper and fit the model.

Use:

- `criterion=nn.MSELoss`
- `optimizer=torch.optim.SGD`
- `optimizer__lr=0.01`
- `max_epochs=100`
- `batch_size=64`
- a fixed `ValidSplit(0.1, random_state=0)`

In [ ]:
torch.manual_seed(0)
semi_parametric = NeuralNetRegressor(
    module=SemiParametricNet,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=...,          # YOUR TASK
    max_epochs=...,             # YOUR TASK
    batch_size=...,             # YOUR TASK
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)

semi_parametric.fit(X_train_house, y_train_house)
print('epochs run:', len(semi_parametric.history))
print('final training RMSE:',
      f"{np.sqrt(semi_parametric.history[-1, 'train_loss']):.4f}")
print('final validation RMSE:',
      f"{np.sqrt(semi_parametric.history[-1, 'valid_loss']):.4f}")

In [ ]:
epochs = np.arange(1, len(semi_parametric.history) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, np.sqrt(semi_parametric.history[:, 'train_loss']),
         label='training RMSE')
plt.plot(epochs, np.sqrt(semi_parametric.history[:, 'valid_loss']),
         label='validation RMSE')
plt.xlabel('epoch')
plt.ylabel('RMSE (target units)')
plt.title('Semi-parametric model: training and validation')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Q2.3 Compare with linear regression and a plain MLP [7 points]

The `PlainMLP` class is provided. Fit it with the same recipe, open the housing test target, and build a table with **test RMSE** and **test MAE** for three models:

- `Linear regression`
- `Plain MLP`
- `Semi-parametric`

A reasonable run should put the semi-parametric model between the linear baseline and the plain MLP, or close to the MLP.

In [ ]:
class PlainMLP(nn.Module):
    def __init__(self):
        super().__init__()
        torch.manual_seed(0)
        self.net = nn.Sequential(
            nn.Linear(8, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)


y_test_house = pd.read_csv(
    f'{HOUSING_BASE_URL}/test.csv', usecols=[target_column]
)[[target_column]].to_numpy(dtype=np.float32)

linear_baseline = LinearRegression().fit(
    X_train_house, y_train_house
)
linear_pred = linear_baseline.predict(X_test_house)

torch.manual_seed(0)
plain_mlp = NeuralNetRegressor(
    module=PlainMLP,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=0.01,
    max_epochs=100,
    batch_size=64,
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)
plain_mlp.fit(X_train_house, y_train_house)
plain_pred = plain_mlp.predict(X_test_house)

semi_pred = semi_parametric.predict(X_test_house)

# YOUR TASK: build a table with test RMSE and test MAE for the three models.
comparison_rows = []
for model_name, prediction in [
    ('Linear regression', linear_pred),
    ('Plain MLP', plain_pred),
    ('Semi-parametric', semi_pred),
]:
    comparison_rows.append({
        'model': model_name,
        'RMSE': ...,
        'MAE': ...,
    })

comparison = pd.DataFrame(comparison_rows).sort_values('RMSE')
display(comparison.round(4))

## Q2.4 Interpret the two parts [5 points]

Answer in two or three sentences:

1. What does the linear branch capture that a plain MLP may need more data to learn?
2. What does the nonlinear branch add?
3. How is this additive structure similar to additive NCF?

**Your answer:**

> Write your answer here.

# Question 3 — A custom recommender [35 points]

Implement the following prediction rule exactly:

$$
\widehat r_{ui}=\mathbf{p}_u^\top\mathbf{q}_i+b_i+\mathbf{p}_u^\top\mathbf{p}_u+a_u.
$$

Here:

- $\mathbf{p}_u$ is the user embedding;
- $\mathbf{q}_i$ is the item embedding;
- $b_i$ is the item bias;
- $a_u$ is the user bias;
- $\mathbf{p}_u^\top\mathbf{p}_u$ is the squared norm of the user vector.

**Note:** the prediction result is not the main point. This question assesses whether you can translate the formula into a correct `forward` pass and train it with skorch. Report RMSE and MAE.

## Q3.1 Complete the custom model [10 points]

**Your task:** complete `CustomRS`.

- Create the two embedding tables and the two bias tables.
- In `forward`, compute the dot product, the user squared norm, and add the two biases.
- Follow the formula exactly; do not add a global mean unless you can justify it.

In [ ]:
class CustomRS(nn.Module):
    def __init__(self, n_users, n_items, K=16):
        super().__init__()
        torch.manual_seed(0)
        # YOUR TASK: user and item embeddings
        self.user_emb = ...
        self.item_emb = ...
        # YOUR TASK: user and item biases
        self.user_bias = ...
        self.item_bias = ...

        # Keep this initialization after you create the tables.
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, x):
        u = x[:, 0].long()
        i = x[:, 1].long()
        pu = self.user_emb(u)
        qi = self.item_emb(i)
        # YOUR TASK: p_u^T q_i
        dot = ...
        # YOUR TASK: p_u^T p_u
        user_norm = ...
        # YOUR TASK: combine exactly as in the formula
        return ...

### Shape check

After completing the class, run the next cell. The output should be `(5, 1)`.

In [ ]:
demo_custom = CustomRS(n_users, n_items)
print(demo_custom)
print('learned numbers:', sum(p.numel() for p in demo_custom.parameters()))
print('output shape:', tuple(demo_custom(torch.as_tensor(X_train[:5])).shape))

## Q3.2 Fit with skorch [10 points]

**Your task:** complete the skorch wrapper and fit the model.

Use:

- `criterion=nn.MSELoss`
- `optimizer=torch.optim.SGD`
- `optimizer__lr=0.05`
- `max_epochs=100`
- `batch_size=256`
- a fixed `ValidSplit(0.1, random_state=0)`

Use 100 epochs here; the custom formula converges more slowly than Q1.

In [ ]:
torch.manual_seed(0)
custom_rs = NeuralNetRegressor(
    module=CustomRS,
    module__n_users=n_users,
    module__n_items=n_items,
    criterion=nn.MSELoss,
    optimizer=torch.optim.SGD,
    optimizer__lr=...,          # YOUR TASK
    max_epochs=...,             # YOUR TASK
    batch_size=...,             # YOUR TASK
    iterator_train__shuffle=True,
    train_split=ValidSplit(0.1, random_state=0),
    verbose=0,
)

custom_rs.fit(X_train, y_train)
print('epochs run:', len(custom_rs.history))
print('final training RMSE:',
      f"{np.sqrt(custom_rs.history[-1, 'train_loss']):.4f}")
print('final validation RMSE:',
      f"{np.sqrt(custom_rs.history[-1, 'valid_loss']):.4f}")

In [ ]:
epochs = np.arange(1, len(custom_rs.history) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, np.sqrt(custom_rs.history[:, 'train_loss']),
         label='training RMSE')
plt.plot(epochs, np.sqrt(custom_rs.history[:, 'valid_loss']),
         label='validation RMSE')
plt.xlabel('epoch')
plt.ylabel('RMSE')
plt.title('Custom recommender: training and validation')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Q3.3 Evaluate and compare [10 points]

**Your task:** predict a rating for every test pair and report **RMSE** and **MAE**. Then compare the custom model with the Q1 two-branch model in one table, sorted by RMSE.

A reasonable run should give RMSE around **1.10–1.20** and MAE around **0.85–0.90**. The custom formula is deliberately unusual, so it may be worse than Q1; the implementation is the point.

In [ ]:
# YOUR TASK: predict a rating for every test pair
custom_predictions = ...

# YOUR TASK: compute RMSE and MAE
custom_rmse = ...
custom_mae = ...

# YOUR TASK: reuse the Q1 RMSE and MAE from above, then build the table
comparison_rs = pd.DataFrame([
    {
        'model': 'Two-branch NCF',
        'RMSE': ...,
        'MAE': ...,
    },
    {
        'model': 'CustomRS',
        'RMSE': custom_rmse,
        'MAE': custom_mae,
    },
]).sort_values('RMSE')

display(comparison_rs.round(4))

## Q3.4 Interpret the custom formula [5 points]

Answer in two or three sentences:

1. What does $\mathbf{p}_u^\top\mathbf{p}_u$ add to the prediction?
2. Does this term depend on the item? What does that imply?
3. Why is the prediction result not the main point of this question?

**Your answer:**

> Write your answer here.

# Submission checklist

Before submitting:

- [ ] replace every `...` in the graded code cells;
- [ ] complete all written responses;
- [ ] run the notebook from top to bottom without an error;
- [ ] keep the cell outputs visible;
- [ ] do not use the test ratings before the evaluation cells;
- [ ] submit the `.ipynb` file through BlackBoard.

**Academic honesty:** the code and written answers must be your own. You may discuss the ideas, but do not share completed code.